# P10.6-AI — Notebook 60: evaluación interna y exportación foraminal

Evalúa una única vez el checkpoint aprobado del Notebook 59 sobre el `internal_test` sellado. Los gates se recuperan del checkpoint antes de calcular test. No utiliza el test oficial.

`humanReviewRequired=true` · `notClinicalDiagnosis=true` · `officialTestAccessed=false`


In [1]:
from __future__ import annotations
import importlib.util, subprocess, sys
required={"pydicom":"pydicom","timm":"timm","tqdm":"tqdm","sklearn":"scikit-learn","kaggle":"kaggle"}
missing=[pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing: subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*missing])
print({"installedNow":missing})


{'installedNow': ['pydicom']}


In [2]:
import json, subprocess, sys
from pathlib import Path
import torch
from google.colab import drive  # type: ignore
if not torch.cuda.is_available(): raise RuntimeError("Seleccioná GPU T4 o superior en Colab.")
print({"gpu":torch.cuda.get_device_name(0),"torch":torch.__version__})
drive.mount("/content/drive",force_remount=False)
REPO_URL="https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"
REPO_REF="enzo/p10-6-ai-rsna-findings"
REPO_ROOT=Path("/content/PFI_MVPTest_Enzo_AImodule")
if not (REPO_ROOT/".git").exists():
    subprocess.check_call(["git","clone","--branch",REPO_REF,"--single-branch",REPO_URL,str(REPO_ROOT)])
else:
    subprocess.check_call(["git","fetch","origin",REPO_REF],cwd=REPO_ROOT)
    subprocess.check_call(["git","checkout",REPO_REF],cwd=REPO_ROOT)
    subprocess.check_call(["git","pull","--ff-only","origin",REPO_REF],cwd=REPO_ROOT)
REPO_SHA=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO_ROOT,text=True).strip()
sys.path.insert(0,str(REPO_ROOT/"ai_service"))
from pfi_ai_service.training.rsna_foraminal_evaluation import evaluate_foraminal_internal
print({"repoRef":REPO_REF,"repoSha":REPO_SHA})


{'gpu': 'Tesla T4', 'torch': '2.11.0+cu128'}
Mounted at /content/drive
{'repoRef': 'enzo/p10-6-ai-rsna-findings', 'repoSha': '627311cb0da57856b30c7b077f95bcb68f54ed94'}


In [3]:
PFI_ROOT=Path("/content/drive/MyDrive/PFI_MVP")
RESULTS_ROOT=PFI_ROOT/"results"/"P10_6_rsna_findings"
MODEL_ROOT=PFI_ROOT/"models"/"P10_6_rsna_findings"/"foraminal_sagittal_t1_2p5d"
summary=evaluate_foraminal_internal(
    split_root=RESULTS_ROOT/"notebook58_foraminal_split",
    training_root=RESULTS_ROOT/"notebook59_foraminal_training",
    evaluation_root=RESULTS_ROOT/"notebook60_foraminal_evaluation",
    model_root=MODEL_ROOT,
    cache_root=Path("/content/rsna_foraminal_internal_test_cache"),
    data_candidates=[Path("/content/RSNA_LUMBAR_DISC"),PFI_ROOT/"data"/"RSNA_LUMBAR_DISC"],
    competition="rsna-2024-lumbar-spine-degenerative-classification",
    local_download_root=Path("/content/RSNA_LUMBAR_DISC"),
    repo_ref=REPO_REF,
    repo_sha=REPO_SHA,
)
print(json.dumps(summary,indent=2,ensure_ascii=False))


cache internal_test por serie:   0%|          | 0/297 [00:00<?, ?it/s]

internal test:   0%|          | 0/93 [00:00<?, ?it/s]

{
  "schemaVersion": "pfi.rsna-foraminal-evaluation.v1",
  "notebook": 60,
  "status": "APPROVED_FOR_NOTEBOOK_61",
  "approved": true,
  "nextNotebook": 61,
  "completedAtUtc": "2026-08-05T01:59:05.826454+00:00",
  "data": {
    "rows": 2955,
    "studies": 296,
    "overlaps": {
      "train": 0,
      "validation": 0
    }
  },
  "internalTestMetrics": {
    "macro_f1": 0.6095263454654657,
    "balanced_accuracy": 0.6439625383934904,
    "normal_mild_recall": 0.8772234273318872,
    "moderate_recall": 0.500945179584121,
    "severe_recall": 0.5537190082644629,
    "weighted_log_loss": 0.621714072809217,
    "runtime_seconds": 16.089608669281006,
    "samples_per_second": 183.65891058878375
  },
  "frozenGates": {
    "macro_f1": 0.36,
    "balanced_accuracy": 0.45,
    "severe_recall": 0.3,
    "moderate_recall": 0.25
  },
  "processGates": {
    "sourceNotebook58Approved": true,
    "sourceNotebook59Approved": true,
    "checkpointHashVerified": true,
    "trainInternalIsolation": t

## Resultado esperado

Una ejecución aprobada finaliza con `APPROVED_FOR_NOTEBOOK_61` y exporta `rsna_foraminal_sagittal_t1_2p5d.pt`. Si falla un gate, conserva la evidencia pero no exporta el modelo final y no debe ajustarse el checkpoint usando el internal test.
